# STAC AI Workshop · NYC · 21 October 2026
## Section 4 · Diagnose the failure before changing the model
**Participant workbook · Matthew Dixon, PhD, FRM**

**Question:** Does a bad action come from the evidence, figure selection, calculation, interpretation or action rule?

This notebook executes a **controlled synthetic workflow**, with deliberately injected faults. Its traces are observable program steps, not reconstructed LLM thoughts. No GPU, API, training or future return is used. The separate recorded LoRA notebook contains the actual measured model comparison.

**Objectives:** trace the first failed observable check; distinguish potentially affected paths from actual changes; apply a verified replacement and rerun; choose a repair before deciding whether adaptation is justified.


In [ ]:
from pathlib import Path
import sys,json
root=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'workflow_diagnosis.py').exists()),None)
if root is None: raise RuntimeError('Open inside the extracted workshop folder.')
sys.path.insert(0,str(root))
import workflow_diagnosis as engine
import adaptation_display as view
experiment=engine.build()
view.panel('Executed exercise · ready',view.table(['Verification','Result'],[[k,v] for k,v in experiment['verification'].items()]),experiment['scope'])


### 1 · Select and inspect a failure
Start with `CASE_ID='period'`. Then compare `basis`, `arguments`, `action`, and `two_faults`.

The execution gate may withhold several cases for different reasons. A shared final action does not imply a shared failure.


In [ ]:
CASE_ID='period'
case=next(c for c in engine.CASES if c['id']==CASE_ID)
before=engine.evaluate(case)
view.panel(case['title'],view.table(['Stage','Observed value','Check'],[[n['label'],before['nodes'][n['id']]['value'],before['nodes'][n['id']]['check']] for n in engine.GRAPH]),'First observable failed check: '+str(before['first_observable_failure']))
view.panel('Declared evidence and policy',view.details('Supplied reference document',experiment['reference_document'])+view.bullets(['Require Harbor Metals, 2026 Q2, reported EPS.','Surprise = (reported EPS − consensus EPS) / |consensus EPS|.','Signal is POSITIVE only above 10%; BUY for POSITIVE, otherwise HOLD.','The gate withholds any proposal when a required observable check fails.']), 'These rules and reference fields are supplied for the exercise. They are not automatically discovered financial truth.')


In [ ]:
view.panel('Dependency graph · inspect the branches',view.table(['Node','Type','Depends on'],[[n['id'],n['type'],', '.join(n['parents']) or 'Independent root'] for n in engine.GRAPH]),'The selected figure also feeds the earnings report. The calculation also feeds a negative-surprise flag. The delegated permission is an independent branch.')


### 2 · Predict what a repair will change
Choose `source`, `selected`, `tool_request`, `recommendation` or `all`.

Before running, write down which checks should pass, which dependent outputs might change, and whether another fault could remain. A replacement fixes only its selected stage; downstream faulty steps are retained.


In [ ]:
REPAIR_ID='source'
my_prediction='Write your prediction before applying the replacement.'
intervention=next(x for x in engine.INTERVENTIONS if x['id']==REPAIR_ID)
result=engine.run(case,intervention)
after=result['after']
view.panel('Controlled intervention · '+intervention['label'],view.table(['Quantity','Computed result'],[['Potential descendants',result['potential_descendants']],['Actually changed nodes',result['actually_changed_nodes']],['Remaining failed checks',result['remaining_failed_checks']],['Proposed action before / after',before['proposed_action']+' / '+after['proposed_action']],['Permitted action before / after',before['terminal_action']+' / '+after['terminal_action']]]),'Replacements are supplied verified exercise values. This is conditional stage localization, not proof of a hidden neural mechanism.')
view.panel('Before and after',view.table(['Node','Before value / check','After value / check'],[[n['id'],{'value':before['nodes'][n['id']]['value'],'check':before['nodes'][n['id']]['check']},{'value':after['nodes'][n['id']]['value'],'check':after['nodes'][n['id']]['check']}] for n in engine.GRAPH]))


### 3 · The upstream repair is not always sufficient
Run the double-fault case below. Correct extraction restores the report, but the incorrect calculator argument still blocks execution. Do not infer “train the model” from a final bad decision.


In [ ]:
double=next(c for c in engine.CASES if c['id']=='two_faults')
partial=engine.run(double,next(i for i in engine.INTERVENTIONS if i['id']=='selected'))
complete=engine.run(double,next(i for i in engine.INTERVENTIONS if i['id']=='all'))
assert partial['remaining_failed_checks']==['tool_request']
assert complete['restored']
view.panel('One repair can expose the next failure',view.table(['Intervention','Remaining failed checks','Execution'],[['Correct extraction only',partial['remaining_failed_checks'],partial['after']['terminal_action']],['Repair all declared stages',complete['remaining_failed_checks'],complete['after']['terminal_action']]]),'The graph makes the unresolved dependency visible. Training cannot fix a calculator call whose arguments are supplied incorrectly by application code.')


### 4 · Choose the remedy, then qualify it
- Wrong period: repair retrieval filters and source eligibility.
- Wrong figure/basis: repair extraction or its checks; consider adaptation only after defining and testing that task.
- Wrong calculator inputs: repair tool argument construction and validation.
- Action violates the signal: enforce the declared action policy; a preference-trained model is not an enforcement boundary.

The independent permission branch remains unchanged in all 30 replay conditions. Changed descendants are computed from the graph; no causal claim is made about unobserved model internals. The five controlled cases are instructional evidence, not a production validation set.

**Exit task:** state the first observable failure, which replacement changed the outcome, what remained unresolved, and why your proposed repair is appropriate. Then open `04_actual_adaptation_readable.ipynb` to inspect the separate measured 16-case LoRA comparison.
